In [1]:
import dolfinx
print(dolfinx.__version__)

[1789462023.981010] [DESKTOP-JTCK1QB:13399:0]         netlink.c:134  UCX  ERROR   failed to send netlink message on fd=92: Input/output error
[1789462023.981061] [DESKTOP-JTCK1QB:13399:0]         netlink.c:134  UCX  ERROR   failed to send netlink message on fd=92: Input/output error
0.11.0


In [2]:
from dolfinx import fem, io
import numpy as np
from mpi4py import MPI
import gmsh
import ufl
from slepc4py import SLEPc
from dolfinx.fem import petsc

In [3]:
# gmsh
gmsh.initialize()
gmsh.model.add("bar")
gmsh.model.occ.importShapes("./cubesat.step")
gmsh.model.occ.synchronize()

# all solids 3d
solids = gmsh.model.getEntities(dim=3)

if len(solids) != 103:
    print("something is wrong in import")

Info    :  - Label 'Shapes/main_assembly (Assembly)/main_assembly:1/main_assembly' (3D)
Info    :  - Label 'Shapes/main_assembly (Assembly)/casing_base:1/casing_base' (3D)
Info    :  - Color (0.627451, 0.627451, 0.627451) (3D & Surfaces)
Info    :  - Label 'Shapes/main_assembly (Assembly)/inner_assemble (Assembly):1/inner_assemble (Assembly)/cover_casette:1/cover_casette' (3D)
Info    :  - Color (0.627451, 0.627451, 0.627451) (3D & Surfaces)
Info    :  - Label 'Shapes/main_assembly (Assembly)/inner_assemble (Assembly):1/inner_assemble (Assembly)/cavity_casette:1/cavity_casette' (3D)
Info    :  - Color (0.627451, 0.627451, 0.627451) (3D & Surfaces)
Info    :  - Label 'Shapes/main_assembly (Assembly)/inner_assemble (Assembly):1/inner_assemble (Assembly)/cover_casette:2/cover_casette' (3D)
Info    :  - Color (0.627451, 0.627451, 0.627451) (3D & Surfaces)
Info    :  - Label 'Shapes/main_assembly (Assembly)/inner_assemble (Assembly):1/inner_assemble (Assembly)/M4_35:2/M4_35/Hexagon Socket H

In [4]:
# finding the tag for different solid
tol = 1e-5
aluminium_parts_point = [
    dict(point=(1.0,1.0,1.0), name="base_stack"),
    dict(point=(0.0,0.0,11.0),name="ll_base_cover"),
    dict(point=(8.0,0.0,25.0),name="l_casette"),
    dict(point=(0.0,0.0,39.0),name="lt_base_cover"),
    dict(point=(0.0,0.0,47.0),name="middle_stack"),
    dict(point=(0.0,0.0,56.0),name="lm_base_cover"),
    dict(point=(8.0,0.0,65.0),namee="m_casette"),
    dict(point=(0.0,0.0,84.0),name="tm_base_cover"),
    dict(point=(0.0,0.0,92.0),name="top_stack"),
    dict(point=(0.0,0.0,101.0),name="tl_base_cover"),
    dict(point=(8.0,0.0,115.0),name="t_casette"),
    dict(point=(0.0,0.0,129.0),name="tt_base_cover"),
    dict(point=(0.0,0.0,137.0),name="top_plate")
]
aluminium_vol_tag = {}
aluminium_tag = []

for part_id,idx_point in enumerate(aluminium_parts_point,start=1):
    found_volume = None

    for dim, volume_tag in solids:
        inside = gmsh.model.isInside(3,volume_tag,list(idx_point["point"]))

        if inside:
            found_volume = volume_tag
            break

    if found_volume is None:
        print(
            f"Warning: part {part_id}"
            f"point {idx_point['point']} is not inside any volume"
        )
    else:
        aluminium_vol_tag[part_id] = found_volume
        aluminium_tag.append(found_volume)

all_tags = [tag for dim,tag in solids]
steel_vol_tag = [tag for tag in all_tags if tag not in aluminium_tag]
print(len(aluminium_tag))
print(len(steel_vol_tag))

13
90


In [5]:
fragmented, volume_map = gmsh.model.occ.fragment(
    solids,[]
)
gmsh.model.occ.synchronize()
gmsh.model.occ.removeAllDuplicates()
gmsh.model.occ.synchronize()

aluminium_new_tag = []
steel_new_tag = []

for i, original_volume in enumerate(solids):
    original_tag = original_volume[1]
    new_entities = volume_map[i]

    for dim,new_tag in new_entities:
        if dim != 3:
            continue

        if original_tag in aluminium_tag:
            aluminium_new_tag.append(new_tag)
        elif original_tag in steel_vol_tag:
            steel_new_tag.append(new_tag)

aluminium_new_tag = sorted(set(aluminium_new_tag))
steel_new_tag = sorted(set(steel_new_tag))



In [6]:
# add physical group
gmsh.model.addPhysicalGroup(3,aluminium_new_tag,1,name="aluminium")
gmsh.model.addPhysicalGroup(3,steel_new_tag,2,name="steel")

2

In [7]:
new_volumes = gmsh.model.getEntities(dim=3)
surface_count = {}
print(len(aluminium_new_tag))
print(len(steel_new_tag))

for dim,volume_tag in new_volumes:
    boundary = gmsh.model.getBoundary(
        [(3,volume_tag)],
        oriented=False,
        recursive=False
    )

    for bdim,surface_tag in boundary:
        if bdim == 2:
            surface_count[surface_tag] = surface_count.get(surface_tag,0) + 1

print(len(surface_count))
interface_surfaces = [tag for tag,count in surface_count.items() if count >= 2]

17
130
2367


In [ ]:
# local mesh refinement
# distance_field = gmsh.model.mesh.field.add("Distance")
# gmsh.model.mesh.field.setNumbers(distance_field,"FacesList",interface_surfaces)
# threshold_field = gmsh.model.mesh.field.add("Threshold")
# gmsh.model.mesh.field.setNumber(threshold_field,"InField",distance_field)

# gmsh.model.mesh.field.setNumber(
#     threshold_field,
#     "SizeMin",
#     0.5
# )

# gmsh.model.mesh.field.setNumber(
#     threshold_field,
#     "SizeMax",
#     3.0
# )

# gmsh.model.mesh.field.setNumber(
#     threshold_field,
#     "DistMin",
#     0.0
# )

# gmsh.model.mesh.field.setNumber(
#     threshold_field,
#     "DistMax",
#     5.0
# )

# gmsh.model.mesh.field.setAsBackgroundMesh(
#     threshold_field
# )

# gmsh.model.mesh.generate(3)

Info    : Meshing 1D...
Info    : [  0%] Meshing curve 714 (Circle)
Info    : [ 10%] Meshing curve 748 (Circle)
Info    : [ 10%] Meshing curve 782 (Circle)
Info    : [ 10%] Meshing curve 816 (Circle)
Info    : [ 10%] Meshing curve 850 (Circle)
Info    : [ 10%] Meshing curve 884 (Circle)
Info    : [ 10%] Meshing curve 885 (Line)
Info    : [ 10%] Meshing curve 886 (BSpline)
Info    : [ 10%] Meshing curve 887 (BSpline)
Info    : [ 10%] Meshing curve 888 (Circle)
Info    : [ 10%] Meshing curve 889 (Circle)
Info    : [ 10%] Meshing curve 890 (Line)
Info    : [ 10%] Meshing curve 891 (Line)
Info    : [ 10%] Meshing curve 892 (BSpline)
Info    : [ 10%] Meshing curve 893 (BSpline)
Info    : [ 10%] Meshing curve 894 (Line)
Info    : [ 10%] Meshing curve 895 (Circle)
Info    : [ 10%] Meshing curve 896 (Line)
Info    : [ 10%] Meshing curve 897 (Line)
Info    : [ 10%] Meshing curve 898 (Circle)
Info    : [ 10%] Meshing curve 899 (Line)
Info    : [ 10%] Meshing curve 900 (Circle)
Info    : [ 10%] M

Info    : [ 10%] Meshing surface 459 (Plane, Frontal-Delaunay)
Info    : [ 10%] Meshing surface 460 (Plane, Frontal-Delaunay)
Info    : [ 10%] Meshing surface 461 (Plane, Frontal-Delaunay)
Info    : [ 10%] Meshing surface 462 (Plane, Frontal-Delaunay)
Info    : [ 10%] Meshing surface 463 (Plane, Frontal-Delaunay)
Info    : [ 10%] Meshing surface 464 (Plane, Frontal-Delaunay)
Info    : [ 10%] Meshing surface 465 (Plane, Frontal-Delaunay)
Info    : [ 10%] Meshing surface 466 (Plane, Frontal-Delaunay)
Info    : [ 10%] Meshing surface 467 (Plane, Frontal-Delaunay)
Info    : [ 10%] Meshing surface 468 (Plane, Frontal-Delaunay)
Info    : [ 10%] Meshing surface 469 (Plane, Frontal-Delaunay)
Info    : [ 10%] Meshing surface 470 (Plane, Frontal-Delaunay)
Info    : [ 10%] Meshing surface 471 (Plane, Frontal-Delaunay)
Info    : [ 10%] Meshing surface 472 (Cone, Frontal-Delaunay)
Info    : [ 10%] Meshing surface 473 (Cone, Frontal-Delaunay)
Info    : [ 10%] Meshing surface 473 (Cone, MeshAdapt)
In

Info    : [ 10%] Meshing surface 555 (Plane, Frontal-Delaunay)
Info    : [ 10%] :-( There are 3 intersections in the 1D mesh (curves 1085 1084 1084)
Info    : [ 10%] 8-| Splitting those edges and trying again - level 0
Info    : [ 10%] :-( There are 3 intersections in the 1D mesh (curves 1084 1084 1085)
Info    : [ 10%] 8-| Splitting those edges and trying again - level 1
Info    : [ 10%] :-( There are 3 intersections in the 1D mesh (curves 1084 1085 1084)
Info    : [ 10%] 8-| Splitting those edges and trying again - level 2
Info    : [ 10%] :-( There are 3 intersections in the 1D mesh (curves 1084 1084 1085)
Info    : [ 10%] 8-| Splitting those edges and trying again - level 3
Info    : [ 10%] :-( There are 3 intersections in the 1D mesh (curves 1084 1085 1084)
Info    : [ 10%] 8-| Splitting those edges and trying again - level 4
Info    : [ 10%] :-( There are 3 intersections in the 1D mesh (curves 1084 1084 1085)
Info    : [ 10%] 8-| Splitting those edges and trying again - level 5
I

Info    : [ 20%] Meshing surface 855 (Plane, Frontal-Delaunay)
Info    : [ 20%] Meshing surface 856 (Plane, Frontal-Delaunay)
Info    : [ 20%] Meshing surface 857 (Plane, Frontal-Delaunay)
Info    : [ 20%] Meshing surface 858 (Plane, Frontal-Delaunay)
Info    : [ 20%] Meshing surface 859 (Plane, Frontal-Delaunay)
Info    : [ 20%] Meshing surface 860 (Plane, Frontal-Delaunay)
Info    : [ 20%] Meshing surface 861 (Cone, Frontal-Delaunay)
Info    : [ 20%] Meshing surface 862 (Cone, Frontal-Delaunay)
Info    : [ 20%] Meshing surface 862 (Cone, MeshAdapt)
Info    : [ 20%] Meshing surface 863 (Cylinder, Frontal-Delaunay)
Info    : [ 20%] Meshing surface 864 (Cylinder, Frontal-Delaunay)
Info    : [ 20%] Meshing surface 865 (Cone, Frontal-Delaunay)
Info    : [ 20%] Meshing surface 866 (Cone, Frontal-Delaunay)
Info    : [ 20%] Meshing surface 866 (Cone, MeshAdapt)
Info    : [ 20%] Serializing surface 866 and refining all its bounding edges
Info    : [ 20%] Meshing surface 867 (Plane, Frontal-De

Info    : [ 30%] Meshing surface 954 (Plane, Frontal-Delaunay)
Info    : [ 30%] Meshing surface 955 (Cone, Frontal-Delaunay)
Info    : [ 30%] Meshing surface 956 (Cone, Frontal-Delaunay)
Info    : [ 30%] Meshing surface 956 (Cone, MeshAdapt)
Info    : [ 30%] Serializing surface 956 and refining all its bounding edges
Info    : [ 30%] Meshing surface 957 (Plane, Frontal-Delaunay)
Info    : [ 30%] Meshing surface 958 (Plane, Frontal-Delaunay)
Info    : [ 30%] Meshing surface 959 (Plane, Frontal-Delaunay)
Info    : [ 30%] Meshing surface 960 (Plane, Frontal-Delaunay)
Info    : [ 30%] Meshing surface 961 (Plane, Frontal-Delaunay)
Info    : [ 30%] Meshing surface 962 (Plane, Frontal-Delaunay)
Info    : [ 30%] Meshing surface 963 (Plane, Frontal-Delaunay)
Info    : [ 30%] Meshing surface 964 (Cone, Frontal-Delaunay)
Info    : [ 30%] Meshing surface 965 (Cone, Frontal-Delaunay)
Info    : [ 30%] Meshing surface 966 (Plane, Frontal-Delaunay)
Info    : [ 30%] Meshing surface 967 (Cone, Frontal-D

...

Info    : [ 70%] Meshing surface 1912 (Plane, Frontal-Delaunay)
Info    : [ 70%] Meshing surface 1913 (Plane, Frontal-Delaunay)
Info    : [ 70%] Meshing surface 1914 (Plane, Frontal-Delaunay)
Info    : [ 70%] Meshing surface 1915 (Plane, Frontal-Delaunay)
Info    : [ 70%] Meshing surface 1916 (Plane, Frontal-Delaunay)
Info    : [ 70%] Meshing surface 1917 (Plane, Frontal-Delaunay)
Info    : [ 70%] Meshing surface 1918 (Plane, Frontal-Delaunay)
Info    : [ 70%] Meshing surface 1919 (Cone, Frontal-Delaunay)
Info    : [ 70%] Meshing surface 1920 (Cone, Frontal-Delaunay)
Info    : [ 70%] Meshing surface 1920 (Cone, MeshAdapt)
Info    : [ 70%] Serializing surface 1920 and refining all its bounding edges
Info    : [ 70%] Meshing surface 1921 (Plane, Frontal-Delaunay)
Info    : [ 70%] Meshing surface 1922 (Cone, Frontal-Delaunay)
Info    : [ 70%] Meshing surface 1922 (Cone, MeshAdapt)
Info    : [ 70%] Serializing surface 1922 and refining all its bounding edges
Info    : [ 70%] Meshing surfac

Info    : [ 70%] Meshing surface 2014 (Cone, MeshAdapt)
Info    : [ 70%] Serializing surface 2014 and refining all its bounding edges
Info    : [ 70%] Meshing surface 2015 (Cone, Frontal-Delaunay)
Info    : [ 70%] Meshing surface 2016 (Plane, Frontal-Delaunay)
Info    : [ 70%] Meshing surface 2017 (Plane, Frontal-Delaunay)
Info    : [ 70%] Meshing surface 2018 (Plane, Frontal-Delaunay)
Info    : [ 70%] Meshing surface 2019 (Plane, Frontal-Delaunay)
Info    : [ 70%] Meshing surface 2020 (Plane, Frontal-Delaunay)
Info    : [ 70%] Meshing surface 2021 (Plane, Frontal-Delaunay)
Info    : [ 70%] Meshing surface 2022 (Cone, Frontal-Delaunay)
Info    : [ 70%] Meshing surface 2023 (Cone, Frontal-Delaunay)
Info    : [ 70%] Meshing surface 2023 (Cone, MeshAdapt)
Info    : [ 70%] Serializing surface 2023 and refining all its bounding edges
Info    : [ 70%] Meshing surface 2024 (Plane, Frontal-Delaunay)
Info    : [ 70%] Meshing surface 2025 (Cone, Frontal-Delaunay)
Info    : [ 70%] Meshing surface

Info    : [ 70%] Serializing surface 2089 and refining all its bounding edges
Info    : [ 70%] Meshing surface 2090 (Plane, Frontal-Delaunay)
Info    : [ 70%] Meshing surface 2091 (Cone, Frontal-Delaunay)
Info    : [ 70%] Meshing surface 2091 (Cone, MeshAdapt)
Info    : [ 70%] Serializing surface 2091 and refining all its bounding edges
Info    : [ 70%] Meshing surface 2092 (Cone, Frontal-Delaunay)
Info    : [ 70%] Meshing surface 2093 (Plane, Frontal-Delaunay)
Info    : [ 70%] Meshing surface 2094 (Plane, Frontal-Delaunay)
Info    : [ 70%] Meshing surface 2095 (Plane, Frontal-Delaunay)
Info    : [ 70%] Meshing surface 2096 (Plane, Frontal-Delaunay)
Info    : [ 70%] Meshing surface 2097 (Plane, Frontal-Delaunay)
Info    : [ 80%] Meshing surface 2098 (Plane, Frontal-Delaunay)
Info    : [ 80%] Meshing surface 2099 (Cone, Frontal-Delaunay)
Info    : [ 80%] Meshing surface 2100 (Cylinder, Frontal-Delaunay)
Info    : [ 80%] Meshing surface 2101 (Plane, Frontal-Delaunay)
Info    : [ 80%] Mes

Error   : Invalid boundary mesh (overlapping facets) on surface 2174 surface 2232


Exception: Invalid boundary mesh (overlapping facets) on surface 2174 surface 2232